# 🐍 Master Asian & Indian Snake Dataset Training (Google Colab GPU)
This notebook automatically:
1. Downloads the **Indian Snakes Dataset** (Cobra, Krait, Russell's Viper, Saw-scaled Viper, Rat Snake, Python, etc.)
2. Uses your trained baseline model to **auto-generate YOLO bounding boxes** for every single species.
3. Combines it with your existing **snake-emula** dataset to create a massive 2,000+ image dataset.
4. Trains YOLOv8 for 100 epochs on the free T4 GPU.
5. Downloads the high-accuracy `best_asian_snake_detector.pt` directly to your PC!

**Ensure GPU is enabled:** `Runtime -> Change runtime type -> T4 GPU`

In [ ]:
# Step 1: Verify GPU & Install dependencies
!nvidia-smi
!pip install ultralytics roboflow

In [ ]:
# Step 2: Download the Indian Snakes Dataset (~630 MB)
import os
if not os.path.exists('Indian-Snakes-Dataset'):
    print('Cloning Indian-Snakes-Dataset from GitHub...')
    !git clone --depth 1 https://github.com/arjun921/Indian-Snakes-Dataset.git
print('Indian Snakes Dataset downloaded!')

In [ ]:
# Step 3: Download your base snake-emula dataset
from roboflow import Roboflow
rf = Roboflow(api_key="nvSMsXnu4b62mWW7RTwG")
project = rf.workspace("aiexample-c8c2y").project("snake-emula")
version = project.version(2)
emula_dataset = version.download("yolov8")

In [ ]:
# Step 4: AI Auto-Labeling and Dataset Merger
# Converts raw species folders into YOLO bounding boxes and merges datasets
import cv2, glob, shutil, random
from pathlib import Path
from ultralytics import YOLO

print('Initializing YOLO auto-labeler with baseline detector...')
detector = YOLO('yolov8n.pt') # Uses pre-trained detector to localize snakes

# Create Master Dataset directory structure
master_dir = Path('master_asian_snakes')
for split in ['train', 'val']:
    (master_dir / 'images' / split).mkdir(parents=True, exist_ok=True)
    (master_dir / 'labels' / split).mkdir(parents=True, exist_ok=True)

# Copy existing snake-emula dataset into master
print('Merging snake-emula dataset...')
for split in ['train', 'valid']:
    target_split = 'val' if split == 'valid' else 'train'
    img_files = glob.glob(f'{emula_dataset.location}/{split}/images/*.*')
    for img_p in img_files:
        lbl_p = img_p.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
        if os.path.exists(lbl_p):
            shutil.copy(img_p, master_dir / 'images' / target_split)
            shutil.copy(lbl_p, master_dir / 'labels' / target_split)

# Auto-label Indian Snakes Dataset
print('Auto-labeling Indian species (Cobra, Krait, Viper, Python, etc.)...')
categories = {'Venomous': 1, 'Non-Venomous': 0}

indian_count = 0
for category, class_id in categories.items():
    cat_path = Path('Indian-Snakes-Dataset') / category
    for species_folder in cat_path.iterdir():
        if not species_folder.is_dir():
            continue
        species_name = species_folder.name
        print(f'Processing {category} -> {species_name}...')
        images = list(species_folder.glob('*.*'))
        random.shuffle(images)
        
        for i, img_path in enumerate(images):
            if img_path.suffix.lower() not in ['.jpg', '.jpeg', '.png']:
                continue
            
            split = 'val' if i % 6 == 0 else 'train'
            dest_img = master_dir / 'images' / split / f'indian_{class_id}_{indian_count}{img_path.suffix}'
            dest_lbl = master_dir / 'labels' / split / f'indian_{class_id}_{indian_count}.txt'
            
            img = cv2.imread(str(img_path))
            if img is None:
                continue
            h, w = img.shape[:2]
            
            # Detect snake bounding box
            preds = detector(img, conf=0.15, verbose=False)
            boxes = preds[0].boxes
            
            if len(boxes) > 0:
                # Use the most confident detected box
                box = boxes[0]
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                xc = ((x1 + x2) / 2) / w
                yc = ((y1 + y2) / 2) / h
                bw = (x2 - x1) / w
                bh = (y2 - y1) / h
            else:
                # Center crop prior if no box found
                xc, yc, bw, bh = 0.5, 0.5, 0.85, 0.85
                
            shutil.copy(str(img_path), str(dest_img))
            with open(dest_lbl, 'w') as f:
                f.write(f'{class_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n')
            indian_count += 1

print(f'Successfully merged and labeled {indian_count} Indian snake images!')
total_train = len(list((master_dir / 'images' / 'train').glob('*.*')))
total_val = len(list((master_dir / 'images' / 'val').glob('*.*')))
print(f'TOTAL MASTER DATASET: {total_train} Train, {total_val} Val images!')

In [ ]:
# Step 5: Generate Master data.yaml configuration
yaml_content = f"""
path: {os.path.abspath('master_asian_snakes')}
train: images/train
val: images/val

nc: 2
names: ['Non-Venomous', 'Venomous']
"""

with open('master_asian_snakes/data.yaml', 'w') as f:
    f.write(yaml_content.strip())
print('data.yaml written successfully!')

In [ ]:
# Step 6: Train YOLOv8 on the Combined Master Dataset (~10-15 mins on GPU)
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data='master_asian_snakes/data.yaml',
    epochs=200,
    patience=50,
    imgsz=640,
    batch=16,
    device=0,
    name='master_asian_snake_detector'
)

In [ ]:
# Step 7: Automatically Download the Master Model
from google.colab import files

best_weights = 'runs/detect/master_asian_snake_detector/weights/best.pt'
files.download(best_weights)
print('Download initiated! Move this file into runs/detect/snake_detector_yolov8n/weights/best.pt on your PC.')